In [1]:
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00:00:0100:01


In [2]:
!pip install -q -U transformers datasets peft bitsandbytes AutoProcessor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 114.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 105.2 MB/s eta 0:00:0000:01


In [3]:
#Imports
from huggingface_hub import login
import os
from datasets import load_dataset
import transformers
import torch
import requests
from PIL import Image
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from peft import get_peft_model, LoraConfig
from transformers import Trainer
from transformers import TrainingArguments
from transformers import PaliGemmaProcessor
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration


In [4]:
#HF LOGIN
HF_TOKEN = os.environ.get("HF_TOKEN")
login(token = HF_TOKEN )

In [5]:
from huggingface_hub import notebook_login

notebook_login()

In [6]:

#Download the dataset
raw_dataset = load_dataset("caglarmert/full_riscm")

# %90 Train, %10 Test
dataset_split = raw_dataset["train"].train_test_split(test_size=0.1, seed=42)
train_set = dataset_split["train"]
test_set = dataset_split["test"]

README.md:   0%|          | 0.00/482 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/384M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/358M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/375M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/44521 [00:00<?, ? examples/s]

In [7]:
train_set

Dataset({
    features: ['image', 'caption_1', 'caption_2', 'caption_3', 'caption_4', 'caption_5'],
    num_rows: 40068
})

In [8]:
device = "cuda"
model_id = "google/paligemma-3b-pt-224"

In [10]:
processor = PaliGemmaProcessor.from_pretrained(model_id)
def collate_fn(examples):
      texts = ["<image><bos>describe en\n" for example in examples]
      labels= [example['caption_1'] for example in examples]
      images = [example["image"].convert("RGB") for example in examples]
      tokens = processor(text=texts, images=images, suffix=labels,
      return_tensors="pt", padding="longest")
      tokens = tokens.to(torch.bfloat16).to(device)
      return tokens

preprocessor_config.json:   0%|          | 0.00/699 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/607 [00:00<?, ?B/s]

In [11]:
model = PaliGemmaForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.bfloat16).to("cuda")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/603 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [12]:
test_set

Dataset({
    features: ['image', 'caption_1', 'caption_2', 'caption_3', 'caption_4', 'caption_5'],
    num_rows: 4453
})

In [13]:
#Sample images and captions for evaluation
sample_indices = [0, 1, 2, 3, 4]
baseline_images = [test_set[i]["image"] for i in sample_indices]

# selecting 1 caption per image for evaluation
baseline_references = [test_set[i]["caption_1"] for i in sample_indices]

print(baseline_references)

['some planes are near several buildings in an airport .', 'A straight freeway goes through the lawn and some cars are driving on the freeway .', 'The roundabout with some exits and entrances is in the commercial area and a lawn is in the middle of the roundabout .', 'The folded mountain consists of a number of ridges and valleys and is covered with some vegetation .', 'A basketball court next to a lake and some trees beside .']


In [14]:
import torch

baseline_predictions = []

# Inference prompt
prompt =["<image><bos>describe en\n"] 

model.to(device)
model.eval()

with torch.no_grad():
    for img in baseline_images:
        inputs = processor(text=prompt, images=img, return_tensors="pt").to(device)
        
        inputs = {k: v.to(torch.bfloat16) if v.dtype == torch.float32 else v for k, v in inputs.items()}
        
        generate_ids = model.generate(**inputs, max_new_tokens=50)
        
        input_len = inputs["input_ids"].shape[-1]
        decoded_output = processor.decode(generate_ids[0][input_len:], skip_special_tokens=True)
        
        baseline_predictions.append(decoded_output.strip())

print("Predictions", baseline_predictions)

Predictions ['Aerial view of the airport', 'I-10 in Houston', 'Satellite view', 'Satellite view', 'Satellite view']


In [15]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [16]:
import evaluate
metric = evaluate.load("bleu") # bleu metric for evaluation

# Score calculation
baseline_score = metric.compute(predictions=baseline_predictions, references=[[r] for r in baseline_references])
print("Baseline Score:", baseline_score)

Baseline Score: {'bleu': 0.0, 'precisions': [0.07142857142857142, 0.0, 0.0, 0.0], 'brevity_penalty': 0.009630143587403526, 'length_ratio': 0.17721518987341772, 'translation_length': 14, 'reference_length': 79}


In [17]:
import gc
import torch

del model         # release the bf16 baseline model
gc.collect()
torch.cuda.empty_cache()

# Verify
print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")

Allocated: 0.01 GB
Reserved:  0.03 GB


In [18]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer
from transformers import BitsAndBytesConfig

#LoRA config
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

bnb_config = BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_compute_dtype=torch.bfloat16
)

#  Training arguments
args = TrainingArguments(
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,        
    warmup_steps=2,
    learning_rate=2e-4,                    
    weight_decay=1e-6,
    adam_beta2=0.999,
    logging_steps=10,
    optim="adamw_8bit",                   
    save_strategy="steps",
    save_steps=500,                        
    save_total_limit=1,
    output_dir="finetuned_paligemma_riscm_small",
    bf16=True,
    dataloader_pin_memory=False,
    report_to=["tensorboard"],
    remove_unused_columns=False,
    max_steps = 800,
)

In [19]:
model = PaliGemmaForConditionalGeneration.from_pretrained(model_id, quantization_config=bnb_config, device_map={"":0})
model = get_peft_model(model, peft_config)

model.print_trainable_parameters()

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/603 [00:00<?, ?it/s]

trainable params: 3,336,192 || all params: 2,926,802,672 || trainable%: 0.1140


In [20]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_set,
    eval_dataset=test_set,
    data_collator=collate_fn
)
print(next(model.parameters()).device) #Check the device



trainer.train()

2026-05-17 08:50:57.647876: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779007857.842324      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779007857.906255      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779007858.349902      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779007858.349934      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779007858.349937      57 computation_placer.cc:177] computation placer alr

cuda:0


Step,Training Loss
10,8.265998
20,6.438967
30,5.233616
40,4.729072
50,4.505174
60,4.192087
70,4.158797
80,3.997738
90,4.129617
100,3.875661


TrainOutput(global_step=800, training_loss=3.0839349675178527, metrics={'train_runtime': 12330.8133, 'train_samples_per_second': 0.519, 'train_steps_per_second': 0.065, 'total_flos': 2.5736205029866176e+16, 'train_loss': 3.0839349675178527, 'epoch': 0.15972846161525406})

In [21]:
model.eval()
print(next(model.parameters()).device)

cuda:0


In [22]:
import torch

finetuned_predictions = []
prompt = ["<image><bos>describe en\n"]

model.eval()
with torch.no_grad():
    for img in baseline_images:
        inputs = processor(text=prompt, images=img, return_tensors="pt").to(device)
        inputs = {k: v.to(torch.bfloat16) if v.dtype == torch.float32 else v for k, v in inputs.items()}
        generate_ids = model.generate(**inputs, max_new_tokens=50)
        input_len = inputs["input_ids"].shape[-1]
        decoded_output = processor.decode(generate_ids[0][input_len:], skip_special_tokens=True)
        finetuned_predictions.append(decoded_output.strip())

print("Fine-tuned predictions:", finetuned_predictions)

Fine-tuned predictions: ['many planes are parked in a row on the airport .', 'The straight freeway goes through the lawn and some cars are driving on the freeway .', 'The roundabout with some exits and entrances is surrounded by some buildings and a lawn is in the middle of the roundabout .', 'The folded mountain consists of some ridges and valleys and some vegetation is on the mountain .', 'A basketball court next to a lake and some trees beside .']


In [23]:
finetuned_score = metric.compute(
    predictions=finetuned_predictions, 
    references=[[r] for r in baseline_references]
)
print("Fine-tuned Score:", finetuned_score)
print("Baseline Score:", baseline_score)  # for comparison

Fine-tuned Score: {'bleu': 0.6729633383141767, 'precisions': [0.810126582278481, 0.7027027027027027, 0.6231884057971014, 0.578125], 'brevity_penalty': 1.0, 'length_ratio': 1.0, 'translation_length': 79, 'reference_length': 79}
Baseline Score: {'bleu': 0.0, 'precisions': [0.07142857142857142, 0.0, 0.0, 0.0], 'brevity_penalty': 0.009630143587403526, 'length_ratio': 0.17721518987341772, 'translation_length': 14, 'reference_length': 79}


In [ ]:
print(f"{'GT':<60} {'BASELINE':<60} {'FINE-TUNED':<60}")
print("-" * 180)
for gt, base, ft in zip(baseline_references, baseline_predictions, finetuned_predictions):
    print(f"{gt[:55]:<60} {base[:55]:<60} {ft[:55]:<60}")

GT                                                           BASELINE                                                     FINE-TUNED                                                  
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
some planes are near several buildings in an airport .       Aerial view of the airport                                   many planes are parked in a row on the airport .            
A straight freeway goes through the lawn and some cars       I-10 in Houston                                              The straight freeway goes through the lawn and some car     
The roundabout with some exits and entrances is in the       Satellite view                                               The roundabout with some exits and entrances is surroun     
The folded mountain consists of a number of ridges and       Satellite view            

In [25]:
import json
results = {
    "baseline_predictions": baseline_predictions,
    "finetuned_predictions": finetuned_predictions,
    "ground_truth": baseline_references,
    "baseline_bleu": baseline_score,
    "finetuned_bleu": finetuned_score,
}
with open("/kaggle/working/results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)
print("Saved.")

Saved.


In [26]:
import torch
from tqdm import tqdm

n_eval = 100
eval_indices = list(range(n_eval))
eval_images = [test_set[i]["image"] for i in eval_indices]
eval_refs = [test_set[i]["caption_1"] for i in eval_indices]

finetuned_preds_large = []
prompt = ["<image><bos>describe en\n"]
model.eval()

with torch.no_grad():
    for img in tqdm(eval_images):
        inputs = processor(text=prompt, images=img, return_tensors="pt").to(device)
        inputs = {k: v.to(torch.bfloat16) if v.dtype == torch.float32 else v for k, v in inputs.items()}
        generate_ids = model.generate(**inputs, max_new_tokens=50)
        input_len = inputs["input_ids"].shape[-1]
        decoded = processor.decode(generate_ids[0][input_len:], skip_special_tokens=True)
        finetuned_preds_large.append(decoded.strip())

ft_score_large = metric.compute(
    predictions=finetuned_preds_large, 
    references=[[r] for r in eval_refs]
)
print("Fine-tuned BLEU on 100 samples:", ft_score_large)

100%|██████████| 100/100 [02:49<00:00,  1.70s/it]

Fine-tuned BLEU on 100 samples: {'bleu': 0.31853548350376465, 'precisions': [0.6001451378809869, 0.37402190923317685, 0.27843803056027167, 0.2189239332096475], 'brevity_penalty': 0.9313523943358435, 'length_ratio': 0.9336043360433605, 'translation_length': 1378, 'reference_length': 1476}


In [27]:
import torch
from tqdm import tqdm
#Evaluating the baseline model with 100 samples from test set
del model
import gc; gc.collect(); torch.cuda.empty_cache()

baseline_model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.bfloat16
).to("cuda")
baseline_model.eval()

baseline_preds_large = []
prompt = ["<image><bos>describe en\n"]
with torch.no_grad():
    for img in tqdm(eval_images):
        inputs = processor(text=prompt, images=img, return_tensors="pt").to(device)
        inputs = {k: v.to(torch.bfloat16) if v.dtype == torch.float32 else v for k, v in inputs.items()}
        generate_ids = baseline_model.generate(**inputs, max_new_tokens=50)
        input_len = inputs["input_ids"].shape[-1]
        decoded = processor.decode(generate_ids[0][input_len:], skip_special_tokens=True)
        baseline_preds_large.append(decoded.strip())

baseline_score_large = metric.compute(
    predictions=baseline_preds_large, 
    references=[[r] for r in eval_refs]
)
print("Baseline BLEU on 100 samples:", baseline_score_large)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/603 [00:00<?, ?it/s]

100%|██████████| 100/100 [01:22<00:00,  1.21it/s]

Baseline BLEU on 100 samples: {'bleu': 0.0, 'precisions': [0.16981132075471697, 0.033950617283950615, 0.0044444444444444444, 0.0], 'brevity_penalty': 0.08364847558278223, 'length_ratio': 0.2872628726287263, 'translation_length': 424, 'reference_length': 1476}


In [29]:
import json

results = {
    "config": {
        "model_id": "google/paligemma-3b-pt-224",
        "lora_r": 8,
        "lora_alpha": 16,
        "lora_targets": ["q_proj", "v_proj", "k_proj", "o_proj"],
        "trainable_params": "11,298,816 (0.39%)",
        "training_steps": 800,
        "effective_batch_size": 4,
        "learning_rate": 2e-4,
        "quantization": "4-bit NF4, bf16 compute",
        "optimizer": "adamw_8bit",
    },
    "qualitative_5_samples": {
        "ground_truth": baseline_references,
        "baseline_predictions": baseline_predictions,
        "finetuned_predictions": finetuned_predictions,
    },
    "quantitative_100_samples": {
        "baseline_bleu": baseline_score_large,
        "finetuned_bleu": ft_score_large,
    },
}

with open("/kaggle/working/results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

print("Saved.")

Saved.


In [30]:
import os
for root, dirs, files in os.walk("/kaggle/working/"):
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / 1e6
        print(f"{size_mb:8.2f} MB  {path}")

    0.00 MB  /kaggle/working/results.json
    0.00 MB  /kaggle/working/finetuned_paligemma_riscm_small/checkpoint-800/scheduler.pt
    0.02 MB  /kaggle/working/finetuned_paligemma_riscm_small/checkpoint-800/trainer_state.json
    0.01 MB  /kaggle/working/finetuned_paligemma_riscm_small/checkpoint-800/training_args.bin
    0.01 MB  /kaggle/working/finetuned_paligemma_riscm_small/checkpoint-800/rng_state.pth
    0.01 MB  /kaggle/working/finetuned_paligemma_riscm_small/checkpoint-800/README.md
   13.39 MB  /kaggle/working/finetuned_paligemma_riscm_small/checkpoint-800/adapter_model.safetensors
    7.55 MB  /kaggle/working/finetuned_paligemma_riscm_small/checkpoint-800/optimizer.pt
    0.00 MB  /kaggle/working/finetuned_paligemma_riscm_small/checkpoint-800/adapter_config.json
    0.02 MB  /kaggle/working/finetuned_paligemma_riscm_small/runs/May17_08-51-13_dc2b985a168f/events.out.tfevents.1779007873.dc2b985a168f.57.0
    0.01 MB  /kaggle/working/.virtual_documents/__notebook_source__.ipynb


In [31]:
!cd /kaggle/working && zip -r submission_artifacts.zip \
    finetuned_paligemma_riscm_small/checkpoint-800/adapter_model.safetensors \
    finetuned_paligemma_riscm_small/checkpoint-800/adapter_config.json \
    finetuned_paligemma_riscm_small/checkpoint-800/README.md \
    results.json
!ls -la /kaggle/working/submission_artifacts.zip

  adding: finetuned_paligemma_riscm_small/checkpoint-800/adapter_model.safetensors (deflated 8%)
  adding: finetuned_paligemma_riscm_small/checkpoint-800/adapter_config.json (deflated 59%)
  adding: finetuned_paligemma_riscm_small/checkpoint-800/README.md (deflated 65%)
  adding: results.json (deflated 62%)
-rw-r--r-- 1 root root 12377028 May 17 13:06 /kaggle/working/submission_artifacts.zip
